# Тест API: регистрация, вход, консультации

Запускайте ячейки **по порядку сверху вниз**.

**Реальные запросы:** бэкенд шлёт код на почту (нужен SMTP в `.env`), после подтверждения в `ACCESS_TOKEN` попадает JWT, затем тестируются `/consultations`.

| Сценарий | Что выполнять |
|----------|----------------|
| **Новый email** | разделы 3а → 3б (код из письма) |
| **Уже есть в БД** | только раздел 4 «Вход» |

`BASE_URL`: бэкенд (по умолчанию `http://localhost:8000`). См. `.env.template` для SMTP и `JWT_SECRET`.


In [35]:
import json
from datetime import datetime, timezone

import requests


In [36]:
BASE_URL = "http://localhost:8000"

ACCESS_TOKEN = None


def print_response(response: requests.Response) -> None:
    print("Status:", response.status_code)
    try:
        data = response.json()
        print("JSON:")
        print(json.dumps(data, ensure_ascii=False, indent=2))
    except Exception:
        print("Text:")
        print(response.text)


def auth_headers() -> dict:
    if not ACCESS_TOKEN:
        return {}
    return {"Authorization": f"Bearer {ACCESS_TOKEN}"}


## 1. Health


In [37]:
print("=== GET /health ===")
health_resp = requests.get(f"{BASE_URL}/health", timeout=10)
print_response(health_resp)


=== GET /health ===
Status: 200
JSON:
{
  "status": "ok"
}


## 2. Учётные данные

Пароль не короче 8 символов. Для **новой** регистрации — email, которого ещё нет в `service_user`.


In [ ]:
TEST_EMAIL = "test@gmail.com"
TEST_PASSWORD = "MySecretPass123"


## 3а. Новый пользователь — отправить код (`POST /auth/register`)

При `409` email уже занят — используйте другой email или раздел 4 «Вход». При `503` проверьте SMTP на бэкенде.


In [ ]:
print("=== POST /auth/register ===")
register_resp = requests.post(
    f"{BASE_URL}/auth/register",
    json={"login": TEST_EMAIL, "password": TEST_PASSWORD},
    timeout=60,
)
print_response(register_resp)

if register_resp.status_code == 503:
    print("\n>>> Настройте SMTP в .env бэкенда.")
elif register_resp.ok:
    print("\n>>> Проверьте почту, возьмите 4 цифры кода. В следующей ячейке задайте CODE_FROM_EMAIL и запустите её.")


## 3б. Подтвердить код (`POST /auth/register/confirm`)

Подставьте код из письма в `CODE_FROM_EMAIL` и выполните ячейку — в `ACCESS_TOKEN` запишется JWT.


In [ ]:
CODE_FROM_EMAIL = "3170"

print("=== POST /auth/register/confirm ===")
confirm_resp = requests.post(
    f"{BASE_URL}/auth/register/confirm",
    json={"login": TEST_EMAIL, "code": CODE_FROM_EMAIL},
    timeout=30,
)
print_response(confirm_resp)

if confirm_resp.ok:
    ACCESS_TOKEN = confirm_resp.json()["access_token"]
    print("\nACCESS_TOKEN сохранён (начало):", ACCESS_TOKEN[:40] + "...")
else:
    print("\nТокен не получен — проверьте код или повторите шаг 3а.")


## 4. Вход (`POST /auth/login`) — если пользователь уже в БД

Если только что прошли 3б, эту ячейку можно не запускать. Если аккаунт уже создан ранее — выполните **только эту** ячейку вместо 3а–3б.


In [ ]:
print("=== POST /auth/login ===")
login_resp = requests.post(
    f"{BASE_URL}/auth/login",
    json={"login": TEST_EMAIL, "password": TEST_PASSWORD},
    timeout=30,
)
print_response(login_resp)

if login_resp.ok:
    ACCESS_TOKEN = login_resp.json()["access_token"]
    print("\nACCESS_TOKEN сохранён (начало):", ACCESS_TOKEN[:40] + "...")
else:
    print("\nВход не удался — проверьте email/пароль или пройдите регистрацию 3а–3б.")


## 5. Consultations (нужен непустой `ACCESS_TOKEN`)


In [ ]:
if not ACCESS_TOKEN:
    raise RuntimeError("Сначала получите токен: шаг 3б или шаг 4.")

print("=== POST /consultations (create) ===")
create_payload = {
    "consultation_datetime": datetime.now(timezone.utc).isoformat(),
    "complaints": "Боль в груди",
    "disease_anamnesis": "Болеет 3 дня",
    "disease_history": "Перенесён инфаркт миокарда",
    "patient_condition": "Удовлетворительное",
    "life_anamnesis": "Хронические заболевания отрицает",
    "bad_habits": "Курение, алкоголь эпизодически",
    "allergy_anamnesis": "Аллергий не отмечает",
    "vital_parameters": "АД 120/80, ЧСС 70",
    "objective_data": "Физикальный осмотр без особенностей",
    "clinical_assessment": "Стабильное состояние",
    "consultation_conclusion": "Состояние удовлетворительное",
    "main_diagnosis": "I21.0 Острый инфаркт миокарда",
    "comorbid_diagnoses": "E11 Сахарный диабет 2 типа",
    "recommendations": "Наблюдение, ЭКГ, контроль АД",
    "prescriptions": "Аспирин, бета-блокаторы",
    "status_protocol": "draft",
}
create_resp = requests.post(
    f"{BASE_URL}/consultations",
    json=create_payload,
    headers=auth_headers(),
    timeout=30,
)
print_response(create_resp)

CREATED_CONSULTATION_ID = None
if create_resp.ok:
    CREATED_CONSULTATION_ID = create_resp.json().get("id")
print("CREATED_CONSULTATION_ID:", CREATED_CONSULTATION_ID)


In [ ]:
if not ACCESS_TOKEN:
    raise RuntimeError("Нет ACCESS_TOKEN")

print("=== GET /consultations (list) ===")
list_resp = requests.get(
    f"{BASE_URL}/consultations",
    headers=auth_headers(),
    timeout=30,
)
print_response(list_resp)


In [ ]:
if not CREATED_CONSULTATION_ID:
    raise RuntimeError("Сначала успешный create")

print("=== GET /consultations/{id} ===")
detail_resp = requests.get(
    f"{BASE_URL}/consultations/{CREATED_CONSULTATION_ID}",
    headers=auth_headers(),
    timeout=30,
)
print_response(detail_resp)


In [ ]:
if not CREATED_CONSULTATION_ID:
    raise RuntimeError("Нет id консультации")

print("=== PUT /consultations/{id} ===")
update_payload = {
    "complaints": "Обновлённые жалобы",
    "recommendations": "Обновлённые рекомендации",
    "status_protocol": "active",
}
update_resp = requests.put(
    f"{BASE_URL}/consultations/{CREATED_CONSULTATION_ID}",
    json=update_payload,
    headers=auth_headers(),
    timeout=30,
)
print_response(update_resp)


In [ ]:
if not CREATED_CONSULTATION_ID:
    raise RuntimeError("Нет id консультации")

print("=== DELETE /consultations/{id} ===")
delete_resp = requests.delete(
    f"{BASE_URL}/consultations/{CREATED_CONSULTATION_ID}",
    headers=auth_headers(),
    timeout=30,
)
print_response(delete_resp)
